# Huffman树


## 1. Huffman 树

Huffman 树，又称**最优二叉树**，是一类带权路径长度最小的二叉树。它最常见的用途是构造 Huffman 编码，从而实现对字符序列的压缩。

为了定义 Huffman 树，需要先明确几个概念：

- **权值（weight）**：节点所代表对象的重要程度。在编码问题中，权值通常是字符出现的频率或次数。
- **路径长度（path length）**：从根节点到某个节点经过的边数。
- **带权路径长度（Weighted Path Length, WPL）**：所有叶子节点的权值与其路径长度乘积之和：

$$
WPL = \sum_{i=1}^{n} w_i l_i
$$

其中 $w_i$ 表示第 $i$ 个叶子节点的权值，$l_i$ 表示该叶子节点到根节点的路径长度。

Huffman 树解决的问题是：给定 $n$ 个权值，构造一棵有 $n$ 个叶子节点的二叉树，使得整棵树的 WPL 最小。由于权值越大的叶子节点越应该靠近根节点，权值越小的叶子节点越可以放在更深的位置，因此 Huffman 树具有很好的“按频率优化路径长度”的性质。

### Huffman 树的构造思想

Huffman 树使用一种典型的**贪心策略**构造：每一步都选择当前权值最小的两棵树合并。具体过程如下：

1. 将每个权值看成一棵只有根节点的二叉树，形成一个森林。
2. 从森林中选出根节点权值最小的两棵树，作为左右子树合并成一棵新树。
3. 新树的根节点权值等于两棵子树根节点权值之和。
4. 将新树放回森林。
5. 重复上述步骤，直到森林中只剩下一棵树，这棵树就是 Huffman 树。

这个算法之所以正确，关键在于：在最优二叉树中，权值最小的两个叶子节点一定可以放在最深层，并且互为兄弟节点。因此每次把最小的两个权值合并，不会破坏整体最优性。

### 时间复杂度

如果每次都在线性表中寻找两个最小权值，构造复杂度为 $O(n^2)$。更常用的做法是使用**最小堆**维护当前森林中的树根，这样每次取出两个最小节点和插入新节点都只需要 $O(\log n)$，总时间复杂度为 $O(n \log n)$。


In [1]:
from heapq import heappop, heappush


class HuffmanNode:
    """Huffman 树节点。"""

    def __init__(self, weight, symbol=None, left=None, right=None):
        self.weight = weight
        self.symbol = symbol
        self.left = left
        self.right = right

    def is_leaf(self):
        return self.left is None and self.right is None


class HuffmanTree:
    """根据符号权值构造 Huffman 树。"""

    def __init__(self, weights):
        if not weights:
            raise ValueError("weights 不能为空")

        for symbol, weight in weights.items():
            if weight <= 0:
                raise ValueError(f"符号 {symbol!r} 的权值必须为正数")

        self.weights = dict(weights)
        self.root = self._build_tree(self.weights)

    def _build_tree(self, weights):
        heap = []
        order = 0

        for symbol, weight in weights.items():
            node = HuffmanNode(weight, symbol=symbol)
            heappush(heap, (weight, order, node))
            order += 1

        while len(heap) > 1:
            left_weight, _, left = heappop(heap)
            right_weight, _, right = heappop(heap)
            parent = HuffmanNode(left_weight + right_weight, left=left, right=right)
            heappush(heap, (parent.weight, order, parent))
            order += 1

        return heap[0][2]

    def weighted_path_length(self):
        """计算 Huffman 树的带权路径长度 WPL。"""

        def dfs(node, depth):
            if node.is_leaf():
                return node.weight * depth
            total = 0
            if node.left is not None:
                total += dfs(node.left, depth + 1)
            if node.right is not None:
                total += dfs(node.right, depth + 1)
            return total

        return dfs(self.root, 0)

    def print_tree(self):
        """以缩进形式打印 Huffman 树，便于观察构造结果。"""

        def dfs(node, indent=""):
            if node.is_leaf():
                print(f"{indent}{node.symbol!r}: {node.weight}")
                return
            print(f"{indent}*: {node.weight}")
            if node.left is not None:
                dfs(node.left, indent + "  ")
            if node.right is not None:
                dfs(node.right, indent + "  ")

        dfs(self.root)


# 示例：根据字符出现次数构造 Huffman 树
weights = {"A": 5, "B": 9, "C": 12, "D": 13, "E": 16, "F": 45}
tree = HuffmanTree(weights)
tree.print_tree()
print("WPL =", tree.weighted_path_length())


*: 100
  'F': 45
  *: 55
    *: 25
      'C': 12
      'D': 13
    *: 30
      *: 14
        'A': 5
        'B': 9
      'E': 16
WPL = 224


## 2. Huffman 编码

Huffman 编码是一种**变长前缀编码**。它利用 Huffman 树为每个符号分配一个二进制编码：

- 从根节点出发，走向左子树记为 `0`；
- 走向右子树记为 `1`；
- 从根节点到某个叶子节点经过的 `0`、`1` 序列，就是该叶子节点对应符号的编码。

由于 Huffman 树中的每个符号都存储在叶子节点上，任何一个符号的编码都不可能是另一个符号编码的前缀。因此 Huffman 编码是一种**前缀码**。前缀码的好处是解码时不需要分隔符：从根节点开始，根据比特串逐位向左或向右走，只要走到叶子节点，就能确定一个完整符号，然后回到根节点继续解码。

### 为什么 Huffman 编码可以压缩数据

固定长度编码会给每个符号分配相同长度的二进制串。例如有 $8$ 种字符时，每个字符至少需要 $3$ 位。但在真实文本中，不同字符出现频率往往差异很大。Huffman 编码让出现频率高的字符拥有更短的编码，让出现频率低的字符使用更长的编码，从而降低整体平均编码长度。

例如，如果字符 `F` 的出现次数远高于其他字符，它在 Huffman 树中通常会更靠近根节点，对应编码也更短。这样在编码长文本时，频繁出现的字符会节省大量空间。

### 编码与解码过程

编码过程：

1. 统计文本中每个符号的出现次数。
2. 根据频率构造 Huffman 树。
3. 遍历 Huffman 树，得到每个符号对应的二进制编码表。
4. 将原文本中的每个符号替换成对应编码，得到完整比特串。

解码过程：

1. 从 Huffman 树的根节点开始。
2. 读取比特串中的下一位：读到 `0` 走左子树，读到 `1` 走右子树。
3. 如果到达叶子节点，就输出该叶子节点对应的符号，并回到根节点。
4. 重复直到比特串读取完毕。

需要注意的是，解码必须使用与编码时相同的 Huffman 树，或者至少使用同一张编码表。否则相同的比特串可能被解释成不同的文本。


In [2]:
class HuffmanCodec:
    """Huffman 编码器与解码器。"""

    def __init__(self, text):
        if not text:
            raise ValueError("text 不能为空")

        self.frequencies = {}
        for ch in text:
            self.frequencies[ch] = self.frequencies.get(ch, 0) + 1
        self.tree = HuffmanTree(self.frequencies)
        self.codes = self._build_codes()
        self.reverse_codes = {code: symbol for symbol, code in self.codes.items()}

    def _build_codes(self):
        codes = {}

        def dfs(node, prefix):
            if node.is_leaf():
                # 只有一种字符时，给它分配编码 0。
                codes[node.symbol] = prefix or "0"
                return
            if node.left is not None:
                dfs(node.left, prefix + "0")
            if node.right is not None:
                dfs(node.right, prefix + "1")

        dfs(self.tree.root, "")
        return codes

    def encode(self, text):
        """将文本编码为 0/1 组成的字符串。"""

        try:
            return "".join(self.codes[ch] for ch in text)
        except KeyError as exc:
            raise ValueError(f"字符 {exc.args[0]!r} 不在当前 Huffman 编码表中") from exc

    def decode(self, bits):
        """将 0/1 字符串解码为原文本。"""

        if any(bit not in "01" for bit in bits):
            raise ValueError("bits 只能包含字符 '0' 和 '1'")

        # 单字符文本的 Huffman 树只有一个节点，需要单独处理。
        if self.tree.root.is_leaf():
            if any(bit != "0" for bit in bits):
                raise ValueError("bits 不是当前 Huffman 树生成的合法编码")
            return self.tree.root.symbol * len(bits)

        result = []
        node = self.tree.root

        for bit in bits:
            node = node.left if bit == "0" else node.right
            if node is None:
                raise ValueError("bits 不是当前 Huffman 树生成的合法编码")
            if node.is_leaf():
                result.append(node.symbol)
                node = self.tree.root

        if node is not self.tree.root:
            raise ValueError("bits 在某个字符编码中间结束，无法完整解码")

        return "".join(result)

    def compression_info(self, text):
        """比较 Huffman 编码和定长编码的大致位数。"""

        encoded_length = len(self.encode(text))
        alphabet_size = len(self.codes)
        fixed_width = max(1, (alphabet_size - 1).bit_length())
        fixed_length = len(text) * fixed_width
        return {
            "alphabet_size": alphabet_size,
            "fixed_width": fixed_width,
            "fixed_length": fixed_length,
            "huffman_length": encoded_length,
            "saved_bits": fixed_length - encoded_length,
        }


# 示例：编码与解码
text = "this is an example for huffman encoding"
codec = HuffmanCodec(text)
encoded = codec.encode(text)
decoded = codec.decode(encoded)

print("编码表:")
for ch, code in sorted(codec.codes.items(), key=lambda item: (len(item[1]), item[0])):
    display_ch = "空格" if ch == " " else ch
    print(f"{display_ch!r}: {code}")

print("\n原文本:", text)
print("编码结果:", encoded)
print("解码结果:", decoded)
print("解码正确:", decoded == text)
print("压缩信息:", codec.compression_info(text))


编码表:
'空格': 101
'n': 000
'a': 1100
'e': 1101
'f': 1110
'i': 1001
'm': 0011
'o': 0100
's': 0010
'c': 10000
'd': 10001
'g': 11110
'h': 11111
'l': 01101
'p': 01100
'r': 01110
't': 01010
'u': 01111
'x': 01011

原文本: this is an example for huffman encoding
编码结果: 0101011111100100101011001001010111000001011101010111100001101100011011101101111001000111010111111011111110111000111100000101110100010000010010001100100011110
解码结果: this is an example for huffman encoding
解码正确: True
压缩信息: {'alphabet_size': 19, 'fixed_width': 5, 'fixed_length': 195, 'huffman_length': 157, 'saved_bits': 38}
